# Getting Started: Gemma 4 Developer Agent

## 1. Environment Configuration and Package Installation

In [3]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

Installing 41 wheels from /kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse...
Wheelhouse installation complete.


## 2. Competition Dataset and Sample Submission

In [116]:
import re
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Official quantized competition models:
#   - 'gemma-4-31b-it-qat-w4a16-ct' (default in sample_submission)
#   - 'gemma-4-12b-it-qat-w4a16-ct'
#   - 'gemma-4-e4b-it-qat-w4a16-ct'
#   - 'gemma-4-e2b-it-qat-w4a16-ct'
MODEL_NAME = 'gemma-4-12b-it-qat-w4a16-ct'

# Copy sample_submission to a writable directory for customization and packaging
SAMPLE_SUBMISSION_SRC = DATA_DIR / 'sample_submission'
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(SAMPLE_SUBMISSION_SRC, AGENT_DIR)

# If switching away from 31B, update all agent YAMLs and remove the 31B-shaped sample LoRA adapters
if MODEL_NAME != 'gemma-4-31b-it-qat-w4a16-ct':
    for yaml_path in [AGENT_DIR / 'agent.yaml', *sorted((AGENT_DIR / 'sub_agents').glob('*.yaml'))]:
        if yaml_path.is_file():
            text = yaml_path.read_text(encoding='utf-8')
            text = re.sub(r'^(\s*model:\s*).*$', rf'\g<1>{MODEL_NAME}', text, flags=re.MULTILINE)
            text = re.sub(r'^\s*adapter:\s*.*\n?', '', text, flags=re.MULTILINE)
            yaml_path.write_text(text, encoding='utf-8')
    shutil.rmtree(AGENT_DIR / 'adapters', ignore_errors=True)

# Apply validated context-window fix (sampling.yaml), budget overrides (eval_config.yaml), and Prompt v3
(AGENT_DIR / 'configs' / 'sampling.yaml').write_text(
    """temperature: 0.2
top_p: 0.95
max_output_tokens: 4096
thinking_config:
  thinking_budget: 2048
  include_thoughts: true
""",
    encoding='utf-8',
)

(AGENT_DIR / 'eval_config.yaml').write_text(
    """evaluation:
  timeout_seconds: 60
  max_tool_calls: 25
  max_time_minutes: 5
  max_turns: 50
""",
    encoding='utf-8',
)

_sys_prompt_path = AGENT_DIR / 'prompts' / 'system.md'
_orig_sys_prompt = _sys_prompt_path.read_text(encoding='utf-8')
_extra_sys_prompt = """
Additional execution requirements:
- Always specify `start_line` and `end_line` (at most 120 lines per call) when calling `read_file` on large files. Never read an entire multi-hundred-line file at once.
- When calling `edit_file`, copy the exact lines from your most recent `read_file` output into `old_string` (including indentation) so edits succeed on the first attempt.
- Follow existing repository conventions for exception types, error messages, and helper/delegation patterns in the surrounding class (including base class methods that may shadow `__getattr__`).
- Reserve at least 3 tool calls before the budget limit for focused verification and `submit_patch`.
- NEVER create or leave untracked scripts (such as `reproduce.py` or `debug.py`) inside `/workspace`, because `submit_patch` runs `git add -N .` and any untracked file in `/workspace` will corrupt your patch and cause verification failure. Always run inline checks via `python3 -c "..."` or `pytest <test_file> -k <pattern>`, or write temporary scripts to `/tmp/repro.py`.
"""
if 'Additional execution requirements:' not in _orig_sys_prompt:
    _sys_prompt_path.write_text(_orig_sys_prompt + '\n' + _extra_sys_prompt, encoding='utf-8')

# Load tasks from the published dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR} (model={MODEL_NAME})')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

Data directory: /kaggle/input/competitions/gemma-4-developer-agent
Agent directory: /kaggle/working/sample_submission (model=gemma-4-12b-it-qat-w4a16-ct)
Loaded 129 tasks from tasks.jsonl
  - fastapi_15661 (fastapi/fastapi @ ee22a4b8)
  - fastapi_15588 (fastapi/fastapi @ cb83b83d)
  - fastapi_15589 (fastapi/fastapi @ 22b02e26)
  - fastapi_15030 (fastapi/fastapi @ 48d58ae3)
  - fastapi_14962 (fastapi/fastapi @ 1e78a36b)


## 3. Code Graph Functions

In [5]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# Load the pre-computed repository code graph and node embeddings for the sample task
repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

# Select a connected symbol and query structural neighbors (callers, callees, definitions)
query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(
    node=query_node,
    graph=repo_graph,
    max_neighbors=10,
)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

# Search for semantically similar code nodes using pre-computed vector embeddings
similar = sg.get_similar_nodes(
    node=query_node,
    repo_name=sample_task.repo,
    k=5,
    graph=repo_graph,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

# Extract an induced subgraph connecting the query symbol and its neighbors
focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

Graph for fastapi/fastapi (fastapi_15661):
  Nodes: 4263, Edges: 2430

Neighbors of 'custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute.get_route_handler' (showing up to 5):
  - fastapi/fastapi;custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute
  - fastapi/fastapi;fastapi.exceptions.HTTPException
  - fastapi/fastapi;fastapi.exceptions.RequestValidationError

Top similar nodes to 'custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute.get_route_handler':
  - custom_request_and_route.tutorial002_an_py310.ValidationErrorLoggingRoute.get_route_handler (similarity=1.0000)
  - fastapi.routing.APIRouter.__init__ (similarity=0.9996)
  - fastapi.applications.FastAPI.setup (similarity=0.9993)
  - custom_request_and_route.tutorial002_an_py310.ValidationErrorLoggingRoute (similarity=0.9991)
  - custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute (similarity=0.9991)

Induced subgraph over 4 focal nodes: 4 nodes, 6 edges


## 4. Start vLLM Server

In [6]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

vLLM server started on http://127.0.0.1:8000/v1 (tp=4)


## 5. Run Phase 1 Inference and Phase 2 Verification

In [ ]:
import asyncio
import concurrent.futures
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    if callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn):
        fn = lambda: coro_or_fn(*args, **kwargs)
    else:
        fn = lambda: coro_or_fn
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


# Read optional evaluation settings from the submission's eval_config.yaml / eval_config.yml
eval_config_file = next(
    (c for name in ('eval_config.yaml', 'eval_config.yml') if (c := AGENT_DIR / name).is_file()),
    None,
)
raw_eval_cfg = (
    yaml.safe_load(eval_config_file.read_text(encoding='utf-8')) or {}
    if eval_config_file is not None
    else {}
)
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)

timeout_seconds = int(eval_section.get('timeout_seconds', 300))
verification_timeout_seconds = int(eval_section.get('verification_timeout_seconds', 300))
max_tool_calls = int(eval_section.get('max_tool_calls', 100))
max_time_minutes = float(eval_section.get('max_time_minutes', 60.0))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None

# Select a small subset of tasks from the published dataset
SAMPLE_TASKS = tasks[:2]
limits, gen_constraints = build_submission_limits()

# Configure Evaluator to run Phase 1 (agent patch generation) and Phase 2 (verification)
# using the subprocess sandbox backend and the submission's eval_config parameters
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    verification_timeout_seconds=verification_timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)

evaluator = Evaluator(eval_config)
predictions = []

for idx, task in enumerate(SAMPLE_TASKS, start=1):
    print(f'[{idx}/{len(SAMPLE_TASKS)}] Evaluating {task.instance_id} ({task.repo})...')
    result = run_sync(
        evaluator.evaluate_task,
        task=task,
        task_index=idx,
        total_tasks=len(SAMPLE_TASKS),
    )
    patch = (result.agent_patch or '').strip()
    predictions.append({'id': task.instance_id, 'prediction': patch if patch else 'NO_PATCH'})
    print(
        f'  -> resolved={result.resolved}, '
        f'exit_code={result.test_exit_code}, '
        f'patch_chars={len(patch)}, '
        f'tool_calls={result.tool_calls}, '
        f'duration={result.duration_seconds:.1f}s'
    )

submission_df = pd.DataFrame(predictions, columns=['id', 'prediction'])
display(submission_df)

## 6. Package Submission Archive

In [118]:
import zipfile
from adk_submission import validate_directory
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

# Validate unpacked submission directory structure, file types, symlinks, and size limits
for dirpath, dirnames, filenames in os.walk(AGENT_DIR, followlinks=False):
    parent = Path(dirpath)
    for name in (*dirnames, *filenames):
        assert not (parent / name).is_symlink(), f'Symlinks are not permitted: {(parent / name).relative_to(AGENT_DIR)}'
validate_directory(AGENT_DIR, limits=limits)

# Create submission.zip with agent.yaml at the root of the archive
zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))

# Validate archive contents against competition constraints
with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f'Archive exceeds 3 GiB limit: {total_size}'
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, f'Disallowed file extension: {info.filename}'

print(f'Created {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MiB)')

Created /kaggle/working/submission.zip (0.00 MiB)


## 7. Server Health and GPU Diagnostics

Verify that the local vLLM inference server responds on port 8000, inspect active worker processes, and check L4 GPU memory allocation before running targeted evaluations.


In [7]:
from collections import deque
from pathlib import Path
from urllib.request import urlopen
import subprocess

try:
    with urlopen('http://127.0.0.1:8000/health', timeout=5) as response:
        print(f'vLLM health check: HTTP {response.status}')
except Exception as exc:
    print(f'vLLM connection error: {type(exc).__name__}: {exc}')

gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print('\nGPU status:')
print(gpu.stdout or gpu.stderr)

logs = list(Path('/tmp').glob('inference_server_8000_*.log'))
if logs:
    latest_log = max(logs, key=lambda p: p.stat().st_mtime)
    print(f'Latest server log: {latest_log}')
    with latest_log.open(encoding='utf-8', errors='replace') as stream:
        print(''.join(deque(stream, maxlen=20)))


vLLM health check: HTTP 200

GPU status:
Wed Nov 18 09:18:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   63C    P0             31W /   72W |   20661MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+------

## 8. Targeted Single-Task Evaluation (`fastapi_15588`, Prompt v1, 20 Tool Calls)

Evaluate `fastapi_15588` (SSE `id` and `event` field validation) under an isolated submission copy with Prompt v1 instructions, a 5-minute time limit, and an expanded budget of 20 counted tool calls.


In [8]:
import asyncio
import concurrent.futures
from dataclasses import replace
import json
from pathlib import Path
import shutil
import tempfile
from urllib.request import urlopen
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    if callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn):
        fn = lambda: coro_or_fn(*args, **kwargs)
    else:
        fn = lambda: coro_or_fn
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


with urlopen('http://127.0.0.1:8000/health', timeout=5) as response:
    print(f'vLLM server health: HTTP {response.status}')

prompt_v1_dir = Path(tempfile.mkdtemp(prefix='sse_prompt_v1_20c_', dir=WORKING_DIR))
prompt_v1_agent = prompt_v1_dir / 'agent'
shutil.copytree(AGENT_DIR, prompt_v1_agent)

prompt_path = prompt_v1_agent / 'prompts' / 'system.md'
original_prompt = prompt_path.read_text(encoding='utf-8')
extra_instructions = """
Additional execution requirements:
- Read files with deliberate line ranges. Avoid rereading the same content.
  Batch related inspections when practical.
- Before changing validation or parsing logic, identify boundary cases
  implied by the specification. Follow existing repository conventions
  for exception types and error messages.
- Reserve at least two counted tool calls for focused verification and
  a possible correction. Prefer a small implementation change over
  spending the entire budget on exploration.
- Verify both accepted and rejected inputs. Tests should challenge the
  implementation, rather than merely repeat its assumptions.
- Put temporary verification scripts under /tmp, outside the repository.
  Submit only changes needed to resolve the issue.
"""
prompt_path.write_text(
    original_prompt + '\n' + extra_instructions,
    encoding='utf-8',
)

eval_config_file = next(
    (c for name in ('eval_config.yaml', 'eval_config.yml') if (c := prompt_v1_agent / name).is_file()),
    None,
)
raw_eval_cfg = (
    yaml.safe_load(eval_config_file.read_text(encoding='utf-8')) or {}
    if eval_config_file is not None
    else {}
)
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)
timeout_seconds = int(eval_section.get('timeout_seconds', 60))
verification_timeout_seconds = int(eval_section.get('verification_timeout_seconds', 300))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else 50

limits, gen_constraints = build_submission_limits()

prompt_v1_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=prompt_v1_dir / 'results',
    submission_dir=prompt_v1_agent,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    verification_timeout_seconds=verification_timeout_seconds,
    max_time_minutes=5.0,
    max_tool_calls=20,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)
prompt_v1_config.budget = replace(
    prompt_v1_config.budget,
    time_minutes=5.0,
    tool_calls=20,
)

assert prompt_v1_config.budget.tool_calls == 20
assert prompt_v1_config.budget.time_minutes == 5.0
print(f'Experiment directory: {prompt_v1_dir}')
print(f'Verified budget: tool_calls={prompt_v1_config.budget.tool_calls}, time_minutes={prompt_v1_config.budget.time_minutes}')

target_task = next(t for t in tasks if t.instance_id == 'fastapi_15588')
prompt_v1_result = run_sync(
    Evaluator(prompt_v1_config).evaluate_task,
    task=target_task,
    task_index=1,
    total_tasks=1,
)

prompt_v1_summary = {
    'task_id': target_task.instance_id,
    'resolved': prompt_v1_result.resolved,
    'error': getattr(prompt_v1_result, 'error', None),
    'exit_code': prompt_v1_result.test_exit_code,
    'patch_chars': len(prompt_v1_result.agent_patch or ''),
    'tool_calls': prompt_v1_result.tool_calls,
    'duration_seconds': prompt_v1_result.duration_seconds,
}
(prompt_v1_dir / 'experiment_summary.json').write_text(
    json.dumps(prompt_v1_summary, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
(prompt_v1_dir / 'agent.patch').write_text(
    prompt_v1_result.agent_patch or '',
    encoding='utf-8',
)

print('\nEvaluation Summary:')
print(json.dumps(prompt_v1_summary, ensure_ascii=False, indent=2))
print('\nGenerated Patch:')
print(prompt_v1_result.agent_patch or 'NO_PATCH')

test_output = getattr(prompt_v1_result, 'test_output', '') or ''
test_log = prompt_v1_config.results_dir / 'test_outputs' / 'fastapi_15588.log'
if not test_output and test_log.is_file():
    test_output = test_log.read_text(encoding='utf-8', errors='replace')

print('\nVerification Test Output (tail):')
print(str(test_output)[-8000:] or 'No test output captured.')


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.AGENT_CONFIG is enabled.
  check_feature_enabled()
/tmp/ipykernel_163/2281941138.py:94: UserWarning: [EXPERIMENTAL] EventsCompactionConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  events_compaction_config=EventsCompactionConfig(


vLLM server health: HTTP 200
Experiment directory: /kaggle/working/sse_prompt_v1_20c_20y88gjl
Verified budget: tool_calls=20, time_minutes=5.0


[   0.0s] 📋 System Prompt: You are an expert autonomous software engineer assigned to resolve an issue in a 
repository efficiently and decisively.
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository fastapi/fastapi.
  [   7.9s] 💭 (376 chars) "Okay, I understand the task. I need to validate Server Sent Event (SSE) fields in 
FastAPI to ensure they don't contain m..."
  [   7.9s] 💬 I will begin by exploring the `fastapi/sse.py` file to understand how Server Sent Events are 
currently handled and where to implement the validation.
  [   7.9s] 📖 read_file (fastapi/sse.py)
  [   7.9s]   ✓ {'status': 'ok', 'filepath': 'fastapi/sse.py', 'content': 'from typing import Annotated, 
Any\n\nfrom annotated_doc impor…
  [  15.5s] 💭 (1764 chars) "The `ServerSentEvent` class is defined in `fastapi/sse.py`. It has fields `data`, 
`raw_data`, `event`, `id`, `retry`, an..."
  [  15.5s] 📖 read_file (fastapi/sse.py)
  [  15.5s]   ✓ {'status': 'ok', 'filepath': 'fastapi/sse.py', 'content': '            """\n            
Pre-serialized data string to us…
  [  34.0s] 💭 (4588 chars) "Wait, I see two `ServerSentEvent` definitions? No, I was reading `format_sse_event` 
arguments.

Let me re-read `fastapi/..."
  [  34.0s] 📖 read_file (fastapi/sse.py)
  [  34.0s]   ✓ {'status': 'ok', 'filepath': 'fastapi/sse.py', 'content': 'from typing import Annotated, 
Any\n\nfrom annotated_doc impor…
  [  45.1s] 💭 (1692 chars) "Okay, I'll add `_check_no_newlines` and update `ServerSentEvent`.

```python
def _check_id_no_null(v: str | None) -> str..."
  [  45.1s] ✏️ edit_file (fastapi/sse.py)
  [  45.1s]   ✓ {'status': 'ok', 'filepath': 'fastapi/sse.py', 'occurrences': 1, 'strategy': 'exact', 'diff': '--- 
a/fastapi/sse.py\n+++…
  [  54.0s] 💭 (1573 chars) "Now I will update the `event` and `id` fields in `ServerSentEvent`.

I'll start with `event`.

```python
    event: Anno..."
  [  54.0s] ✏️ edit_file (fastapi/sse.py)
  [  54.1s]   ✓ {'status': 'ok', 'filepath': 'fastapi/sse.py', 'occurrences': 1, 'strategy': 'exact', 'diff': '--- 
a/fastapi/sse.py\n+++…
  [  60.7s] 💭 (792 chars) "Wait, the `old_string` I used in `edit_file` for `event` was slightly different from 
what I saw in the first `read_file`..."
  [  60.7s] ✏️ edit_file (fastapi/sse.py)
  [  60.7s]   ❌ {'status': 'error', 'error_type': 'FileEditError', 'error_message': "Failed to replace: old_string
not found. Ensure you…
  [  61.9s] 💭 (127 chars) "Ah, it seems I have a mismatch in the `old_string` for `id`. Let me read the file again
to see the exact content of `id`..."
  [  61.9s] 📖 read_file (fastapi/sse.py)
  [  62.0s]   ✓ {'status': 'ok', 'filepath': 'fastapi/sse.py', 'content': '            Maps to 
`addEventListener(event, ...)` on the bro…
  [  69.5s] 💭 (960 chars) "The `id` field in `ServerSentEvent` is:

```python
    id: Annotated[
        str | None,
        AfterValidator(_check_..."
  [  69.5s] ✏️ edit_file (fastapi/sse.py)
  [  69.5s]   ❌ {'status': 'error', 'error_type': 'FileEditError', 'error_message': "Failed to replace: old_string
not found. Ensure you…
  [  72.7s] 💭 (281 chars) "It seems I'm having trouble matching the `old_string`. I'll try to use a smaller part 
of it.

```python
    id: Annotate..."
  [  72.7s] ✏️ edit_file (fastapi/sse.py)
  [  72.8s]   ✓ {'status': 'ok', 'filepath': 'fastapi/sse.py', 'occurrences': 1, 'strategy': 'exact', 'diff': '--- 
a/fastapi/sse.py\n+++…
  [  76.7s] 💭 (773 chars) "Okay, that worked. Now I should check if I need to do anything else.

I have:
1. Added `_check_no_newlines` validator.
2..."
  [  76.7s] 🔧 run_command find tests -name "*sse*"
  [  76.7s]   → tests/test_sse.py
  [  77.4s] 💭 (31 chars) "I'll check `tests/test_sse.py`."
  [  77.4s] 📖 read_file (tests/test_sse.py)
  [  77.5s]   ✓ {'status': 'ok', 'filepath': 'tests/test_sse.py', 'content': 'import asyncio\nimport time\nfrom 
collections.abc import A…
  [  84.2s] 💭 (387 chars) "The existing tests don't seem to cover the multi-line `id` or `event` case.


Evaluation Summary:
{
  "task_id": "fastapi_15588",
  "resolved": false,
  "error": null,
  "exit_code": 1,
  "patch_chars": 974,
  "tool_calls": 20,
  "duration_seconds": 220.37450904000002
}

Generated Patch:
diff --git a/fastapi/sse.py b/fastapi/sse.py
index 901d82496..81c040b1a 100644
--- a/fastapi/sse.py
+++ b/fastapi/sse.py
@@ -39,6 +39,12 @@ def _check_id_no_null(v: str | None) -> str | None:
     return v
 
 
+def _check_no_newlines(v: str | None) -> str | None:
+    if v is not None and ("\n" in v or "\r" in v):
+        raise ValueError("SSE 'id' or 'event' must not contain multi-line content")
+    return v
+
+
 class ServerSentEvent(BaseModel):
     """Represents a single Server-Sent Event.
 
@@ -86,6 +92,7 @@ class ServerSentEvent(BaseModel):
     ] = None
     event: Annotated[
         str | None,
+        AfterValidator(_check_no_newlines),
         Doc(
             """
             Optional event type name.
@@ -98,6 +105,7 @@ class ServerSentEvent(BaseModel):
     id

## 9. Multi-Task Benchmark (`Prompt v2` + `sampling.yaml` Context Fix + `SubprocessManager` `src/` Path Fix, 20 Tool Calls)

1. **`configs/sampling.yaml` Context Window Fix**:
   - The starter `max_output_tokens: 16384` against vLLM's `max_model_len=32768` restricts the usable input window to `16,384` tokens, causing `ContextWindowExceededError` whenever the agent inspects a large file before `compaction_interval=15` triggers.
   - Lowering `max_output_tokens` to `4096` and `thinking_budget` to `2048` expands the usable input window to **28,672 tokens** and eliminates mid-trajectory context crashes.
2. **Kaggle `SubprocessManager` `src/` Layout Fix**:
   - In Kaggle's offline `sandbox='subprocess'` environment, `SubprocessManager.exec` sets `PYTHONPATH=/workspace`, but `psf/requests` and `Textualize/rich` use a `src/` layout (`/workspace/src/<pkg>`), causing `import requests` / `import rich` inside the sandbox to import the unedited system package in `/usr/local/lib/python3.12/dist-packages/` instead of `/workspace/src/`.
   - Prepending `/workspace/src` to `PYTHONPATH` in `SubprocessManager.exec` (when `/workspace/src` exists) ensures both Phase 1 agent commands and Phase 2 verification import the edited workspace code.


In [111]:
from dataclasses import replace
import json
from pathlib import Path
import shutil
import tempfile
from urllib.request import urlopen
import swegemma.sandbox.subprocess as sp_mod

with urlopen('http://127.0.0.1:8000/health', timeout=5) as response:
    print(f'vLLM server health: HTTP {response.status}')

# Fix Kaggle SubprocessManager PYTHONPATH for src/-layout repositories (psf/requests, Textualize/rich)
if not getattr(sp_mod.SubprocessManager, '_src_path_patched', False):
    _orig_sp_exec = sp_mod.SubprocessManager.exec

    def _patched_sp_exec(self, sandbox_id, command, *, timeout=None):
        with self._lock:
            ws = self._sandboxes[sandbox_id]['workspace']
        src_dir = ws / 'src'
        if src_dir.is_dir():
            command = f'export PYTHONPATH={src_dir}:{ws}:${{PYTHONPATH:-}}; ' + command
        return _orig_sp_exec(self, sandbox_id, command, timeout=timeout)

    sp_mod.SubprocessManager.exec = _patched_sp_exec
    sp_mod.SubprocessManager._src_path_patched = True

benchmark_dir = Path(tempfile.mkdtemp(prefix='bench_v3_fixed_', dir=WORKING_DIR))
benchmark_agent = benchmark_dir / 'agent'
shutil.copytree(AGENT_DIR, benchmark_agent)

# 1. Fix sampling.yaml: lower max_output_tokens from 16384 -> 4096 so input window expands from 16384 -> 28672 tokens
sampling_path = benchmark_agent / 'configs' / 'sampling.yaml'
sampling_yaml = """temperature: 0.2
top_p: 0.95
max_output_tokens: 4096
thinking_config:
  thinking_budget: 2048
  include_thoughts: true
"""
sampling_path.write_text(sampling_yaml, encoding='utf-8')

# 2. Update eval_config.yaml inside the agent directory
eval_cfg_path = benchmark_agent / 'eval_config.yaml'
eval_cfg_yaml = """evaluation:
  timeout_seconds: 60
  max_tool_calls: 25
  max_time_minutes: 5
  max_turns: 50
"""
eval_cfg_path.write_text(eval_cfg_yaml, encoding='utf-8')

# 3. Apply Prompt v3 instructions to prompts/system.md
prompt_path = benchmark_agent / 'prompts' / 'system.md'
original_prompt = prompt_path.read_text(encoding='utf-8')
prompt_v3_instructions = """
Additional execution requirements:
- Always specify `start_line` and `end_line` (at most 120 lines per call) when calling `read_file` on large files. Never read an entire multi-hundred-line file at once.
- When calling `edit_file`, copy the exact lines from your most recent `read_file` output into `old_string` (including indentation) so edits succeed on the first attempt.
- Follow existing repository conventions for exception types, error messages, and helper/delegation patterns in the surrounding class (including base class methods that may shadow `__getattr__`).
- Reserve at least 3 tool calls before the budget limit for focused verification and `submit_patch`.
- NEVER create or leave untracked scripts (such as `reproduce.py` or `debug.py`) inside `/workspace`, because `submit_patch` runs `git add -N .` and any untracked file in `/workspace` will corrupt your patch and cause verification failure. Always run inline checks via `python3 -c "..."` or `pytest <test_file> -k <pattern>`, or write temporary scripts to `/tmp/repro.py`.
"""
prompt_path.write_text(
    original_prompt + '\n' + prompt_v3_instructions,
    encoding='utf-8',
)

benchmark_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=benchmark_dir / 'results',
    submission_dir=benchmark_agent,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    verification_timeout_seconds=verification_timeout_seconds,
    max_time_minutes=5.0,
    max_tool_calls=25,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=18432,
        event_retention_size=8,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)
benchmark_config.budget = replace(
    benchmark_config.budget,
    time_minutes=5.0,
    tool_calls=25,
)
assert benchmark_config.budget.tool_calls == 25

target_ids = ('requests_7502', 'requests_7433', 'requests_7315', 'rich_4077')
bench_tasks = [t for t in tasks if t.instance_id in target_ids]
bench_evaluator = Evaluator(benchmark_config)
bench_records = []

print(f'Benchmark directory: {benchmark_dir}')
for idx, task in enumerate(bench_tasks, start=1):
    print(f'\n[{idx}/{len(bench_tasks)}] Evaluating {task.instance_id} ({task.repo})...')
    res = run_sync(
        bench_evaluator.evaluate_task,
        task=task,
        task_index=idx,
        total_tasks=len(bench_tasks),
    )
    rec = {
        'task_id': task.instance_id,
        'repo': task.repo,
        'resolved': res.resolved,
        'error': getattr(res, 'error_message', None) or getattr(res, 'error', None),
        'exit_code': res.test_exit_code,
        'patch_chars': len(res.agent_patch or ''),
        'tool_calls': res.tool_calls,
        'duration_seconds': round(res.duration_seconds, 2),
    }
    bench_records.append(rec)
    (benchmark_dir / f'{task.instance_id}.patch').write_text(
        res.agent_patch or '', encoding='utf-8'
    )
    print('  ->', json.dumps(rec, ensure_ascii=False))

(benchmark_dir / 'benchmark_summary.json').write_text(
    json.dumps(bench_records, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
resolved_count = sum(1 for r in bench_records if r['resolved'])
print(f'\nBenchmark complete: {resolved_count}/{len(bench_records)} resolved.')
print(json.dumps(bench_records, ensure_ascii=False, indent=2))


/tmp/ipykernel_163/1228548408.py:84: UserWarning: [EXPERIMENTAL] EventsCompactionConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  events_compaction_config=EventsCompactionConfig(


vLLM server health: HTTP 200
Benchmark directory: /kaggle/working/bench_v3_fixed_2lcir01c

[1/4] Evaluating requests_7502 (psf/requests)...


[   0.0s] 📋 System Prompt: You are an expert autonomous software engineer assigned to resolve an issue in a 
repository efficiently and decisively.
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository psf/requests.
  [   3.2s] 💭 (362 chars) "Okay, I understand the task. I need to fix `_encode_files` detection for 
`__getattr__`-based file wrappers in the `reque..."
  [   3.2s] 💬 I will start by searching for the `_encode_files` function in the codebase to understand its current
implementation and how it handles file wrappers.
...
  [   3.2s] 🔧 run_command grep -r "_encode_files" src/requests
  [   3.2s]   → src/requests/models.py:    def _encode_files(
  [   4.1s] 💭 (113 chars) "Okay, the `_encode_files` function is in `src/requests/models.py`. I'll read that file 
to see the implementation."
  [   4.1s] 📖 read_file (src/requests/models.py)
  [   4.1s]   ✓ {'status': 'ok', 'filepath': 'src/requests/models.py', 'content': 
'"""\nrequests.models\n~~~~~~~~~~~~~~~\n\nThis module …
  [   5.4s] 💭 (93 chars) "The file is long. I need to find `_encode_files`. I'll search for it in the rest of the 
file."
  [   5.4s] 📖 read_file (src/requests/models.py)
  [   5.4s]   ✓ {'status': 'ok', 'filepath': 'src/requests/models.py', 'content': '    @staticmethod\n    def 
_encode_params(\n        d…
  [  19.7s] 💭 (3705 chars) "I've found the `_encode_files` method in `src/requests/models.py`.

```python
    @staticmethod
    def _encode_files(
 ..."
  [  19.7s] 📖 read_file (src/requests/_types.py)
  [  19.7s]   ✓ {'status': 'ok', 'filepath': 'src/requests/_types.py', 'content': 
'"""\nrequests._types\n~~~~~~~~~~~~~~~\n\nThis module …
  [  39.7s] 💭 (4608 chars) "`SupportsRead` is indeed a `runtime_checkable` Protocol.

```python
@runtime_checkable
class SupportsRead(Protocol[_T_co..."
  [  39.7s] 🔧 run_command grep -n "__getattr__" src/requests/models.py
  [  39.7s]   → error (exit 0): 
  [  48.7s] 💭 (2055 chars) "Okay, `__getattr__` is not in `src/requests/models.py`.

Let's look at the `_encode_files` method again.

```python
    ..."
  [  48.7s] 🔧 run_command grep -r "_SupportsRead" src/requests
  [  48.7s]   → src/requests/models.py:from ._types import SupportsRead as _SupportsRead
  [  56.9s] 💭 (1876 chars) "Wait, I see `isinstance(data, _SupportsRead)` in `_encode_params` too.

```python
    @staticmethod
    def _encode_para..."
  [  56.9s] 🔧 run_command find tests -name "*_encode_files*"
  [  56.9s]   🔧 
  [  57.9s] 💭 (106 chars) "Okay, no tests with `_encode_files` in the name. I'll search for `_encode_files` in the
`tests` directory."
  [  57.9s] 🔧 run_command grep -r "_encode_files" tests
  [  57.9s]   → error (exit 0): 
  [  58.7s] 💭 (61 chars) "That's unexpected. Let me try `grep -r "encode_files" tests`."
  [  58.7s] 🔧 run_command grep -r "encode_files" tests
  [  58.8s]   → error (exit 0): 
  [  61.0s] 💭 (348 chars) "Wait, why is `grep` failing? Ah, I might not have permission or it's not working as 
expected. Let me try `find tests -na..."
  [  61.0s] 🔧 run_command find tests -name "*test_*.py" | xargs grep -l "Request.post"
  [  61.0s]   → error (exit 0): 
  [  62.1s] 💭 (152 chars) "Okay, `grep` is failing for some reason. I'll use `find` to list files and then 
`read_file` them if needed.

Let's try t..."
  [  62.1s] 🔧 run_command find tests -name "*files*"
  [  62.2s]   🔧 
  [  63.0s] 💭 (91 chars) "Okay, no tests with "files" in the name.

Let's try to find tests that involve `multipart`."
  [  63.0s] 🔧 run_command find tests -name "*multipart*"
  [  63.1s]   🔧 
  [  63.8s] 💭 (60 chars) "Still nothing.

Let's try to find tests that involve `post`."
  [  63.8s] 🔧 run_command find tests -name "*post*"
  [  63.8s]   🔧 
  [  65.7s] 💭 (274 chars) "Okay, I'll just look for any test file that might be relevant.

Let's try to find where `_encode_files` is called in `sr..."
  [  65.7s] 📖 read_file (src/requests/models.py)
  [  65.7s]   ✓ {'status': 'ok', 'filepath': 'src/requests/models.py', 'content'

  -> {"task_id": "requests_7502", "repo": "psf/requests", "resolved": false, "error": null, "exit_code": 1, "patch_chars": 1153, "tool_calls": 25, "duration_seconds": 275.37}

[2/4] Evaluating requests_7433 (psf/requests)...


[   0.0s] 📋 System Prompt: You are an expert autonomous software engineer assigned to resolve an issue in a 
repository efficiently and decisively.
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository psf/requests.
  [   3.0s] 💭 (604 chars) "The user wants me to fix a bug in `requests` library where `prepare_body` fails to 
detect some stream-like objects that ..."
  [   3.0s] 🔧 run_command grep -r "prepare_body" src/requests | head -n 20
  [   3.0s]   → src/requests/models.py:        self.prepare_body(data, files, json)
  [   4.0s] 💭 (116 chars) "Okay, `prepare_body` is in `src/requests/models.py`.
Let me read `src/requests/models.py` to find the relevant part."
  [   4.0s] 📖 read_file (src/requests/models.py)
  [   4.0s]   ✓ {'status': 'ok', 'filepath': 'src/requests/models.py', 'content': 
'"""\nrequests.models\n~~~~~~~~~~~~~~~\n\nThis module …
  [   5.3s] 💭 (84 chars) "I need to find `prepare_body` in `src/requests/models.py`. It's likely further down."
  [   5.3s] 📖 read_file (src/requests/models.py)
  [   5.3s]   ✓ {'status': 'ok', 'filepath': 'src/requests/models.py', 'content': '    @staticmethod\n    def 
_encode_params(\n        d…
  [   6.7s] 💭 (80 chars) "I'm still looking for `prepare_body`. It's probably a method of `Request` class."
  [   6.7s] 📖 read_file (src/requests/models.py)
  [   6.7s]   ✓ {'status': 'ok', 'filepath': 'src/requests/models.py', 'content': '    :param hooks: dictionary of 
callback hooks, for i…
  [   8.5s] 💭 (169 chars) "I see `self.prepare_body(data, files, json)` in `PreparedRequest.prepare`.
I need to find `prepare_body` method in `Prep..."
  [   8.5s] 📖 read_file (src/requests/models.py)
  [   8.6s]   ✓ {'status': 'ok', 'filepath': 'src/requests/models.py', 'content': '    def __repr__(self) -> str:\n
return f"<Pre…
  [  15.8s] 💭 (1509 chars) "I found `prepare_body` in `src/requests/models.py`.
The relevant part is:

```python
    def prepare_body(
        self,..."
  [  15.8s] 📖 read_file (src/requests/models.py)
  [  15.8s]   ✓ {'status': 'ok', 'filepath': 'src/requests/models.py', 'content': '            netloc += 
f":{port}"\n\n        # Bare do…
  [  21.6s] 💭 (830 chars) "The code is:

```python
        if isinstance(data, Iterable) and not isinstance(
            data, (str, bytes, list, t..."
  [  21.6s] ✏️ edit_file (src/requests/models.py)
  [  21.6s]   ✓ {'status': 'ok', 'filepath': 'src/requests/models.py', 'occurrences': 1, 'strategy': 'exact', 
'diff': '--- a/src/request…
  [  22.9s] 💭 (149 chars) "I have applied the fix. Now I need to verify it.
I'll look for tests that might be relevant. I'll search for `prepare_bo..."
  [  22.9s] 🔧 run_command find tests -name "*prepare_body*"
  [  22.9s]   🔧 
  [  23.7s] 💭 (74 chars) "Okay, that didn't find anything. Let me search for `models.py` in `tests`."
  [  23.7s] 🔧 run_command find tests -name "*models*"
  [  23.7s]   🔧 
  [  24.8s] 💭 (138 chars) "Let me search for `Iterable` in `tests` to see if there are any tests related to 
streaming.
Actually, I'll search for `s..."
  [  24.8s] 🔧 run_command find tests -name "*stream*"
  [  24.8s]   🔧 
  [  25.8s] 💭 (113 chars) "Let me try to find tests that use `Request` and `data` as a stream.
I'll search for `PreparedRequest` in `tests`."
  [  25.8s] 🔧 run_command find tests -name "*PreparedRequest*"
  [  25.8s]   🔧 
  [  26.9s] 💭 (96 chars) "Okay, let me try to find tests that use `data` in `Request`.
I'll search for `data=` in `tests`."
  [  26.9s] 🔧 run_command grep -r "data=" tests | head -n 20
  [  26.9s]   → tests/test_lowlevel.py:        r = requests.post(url, data=data, stream=True)
  [  29.7s] 💭 (539 chars) "`tests/test_requests.py` seems like a good place to look.
I'll check `tests/test_requests.py` to see if there are any te..."
  [  29.7s] 🔧 run_command grep -r "__getattr__" tests
  [  29.7s]   → error (exit 0): 
  [  30.6s] 💭 (68 chars) "Okay, no `__getattr__` in tests.
Let me search for "proxy" in tests."
  [  30.6s] 🔧 run_command grep -

KeyboardInterrupt: 